# 🔁 Step 5 of RAG — Re-ranking

So far our pipeline retrieves chunks with **ensemble retrieval + RRF** and feeds them straight to the LLM. That works well, but the order of chunks still matters: most LLMs pay more attention to context near the top of the prompt.

**Re-ranking** adds a lightweight second-pass step that re-orders the retrieved chunks *before* they reach the LLM:

```
Query  ──►  Ensemble Retrieval (top-K chunks)
                     │
                     ▼
              Re-ranker  ←── Query
                     │
                     ▼
           Top-N re-ordered chunks
                     │
                     ▼
             LangChain RAG Chain
                     │
                     ▼
                  Answer ✅
```

> 💡 **Why does this improve quality without adding latency?**  
> The initial retriever must be fast, so it trades precision for speed. A re-ranker operates over a small shortlist (e.g. 10–20 chunks) and can afford to be more expensive per chunk because it only runs on those few candidates — not the entire corpus.

## 0. Setup & Imports

In [ ]:
import os
import json
import textwrap
import numpy as np
import psycopg2
from pathlib import Path
from dotenv import load_dotenv
from munch import Munch

# Embedding & re-ranking models
from FlagEmbedding import BGEM3FlagModel

# LangChain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# Load config
config_path = (Path.cwd() / "../config/config.yaml").resolve()
with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

# Load secrets
load_dotenv((Path.cwd() / "../.env").resolve())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")

if not SUPABASE_PASSWORD:
    raise ValueError("SUPABASE_PASSWORD not found in .env")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

DATABASE_URL = (
    f"postgresql://{config.supabase.user}:{SUPABASE_PASSWORD}@"
    f"{config.supabase.host}:{config.supabase.port}/{config.supabase.database}"
)
sql_dir = (Path.cwd() / "sql").resolve()

print("✅ Config and secrets loaded")

### Load models

We load two models:
- **BGE-M3** — the same embedding model used throughout this course, for retrieval
- **BGE-Reranker-v2-m3** — a cross-encoder from the same BAAI family, for re-ranking

A **cross-encoder** takes `(query, document)` as a single input and outputs a relevance score, giving it much richer signal than a bi-encoder (which embeds query and document independently).

In [ ]:
print(f"Loading embedding model: {config.embedding.model} …")
embed_model = BGEM3FlagModel(config.embedding.model, use_fp16=True)
print("✅ Embedding model loaded")

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print(f"Loading reranker: {config.reranker.model} …")
reranker_tokenizer = AutoTokenizer.from_pretrained(config.reranker.model)
reranker_model = AutoModelForSequenceClassification.from_pretrained(config.reranker.model)
reranker_model.eval()
print("✅ Reranker loaded")

---

## 1. Ensemble Retrieval (from Notebook 5)

We reuse the same retrieval stack: dense + sparse + ColBERT, fused with Reciprocal Rank Fusion. The only difference here is that we retrieve a **larger candidate pool** (`top_k * reranker_factor`) so the re-ranker has more candidates to work with.

In [ ]:
def get_connection() -> psycopg2.extensions.connection:
    return psycopg2.connect(DATABASE_URL)


def retrieve_dense(q_dense: np.ndarray, top_k: int) -> list[dict]:
    sql = (sql_dir / "09_query_dense.sql").read_text()
    vec_str = f"[{','.join(map(str, q_dense.tolist()))}]"
    conn = get_connection()
    cur  = conn.cursor()
    try:
        cur.execute(sql, (vec_str, vec_str, top_k))
        return [{"chunk_id": r[0], "chunk_text": r[1], "score": float(r[2])} for r in cur.fetchall()]
    finally:
        cur.close(); conn.close()


def retrieve_sparse(q_sparse: dict, top_k: int) -> list[dict]:
    sql = (sql_dir / "10_query_sparse.sql").read_text()
    sparse_json = json.dumps({str(k): float(v) for k, v in q_sparse.items()})
    conn = get_connection()
    cur  = conn.cursor()
    try:
        cur.execute(sql, (sparse_json, top_k))
        return [{"chunk_id": r[0], "chunk_text": r[1], "score": float(r[2])} for r in cur.fetchall()]
    finally:
        cur.close(); conn.close()


def retrieve_colbert(q_colbert: np.ndarray, top_k: int) -> list[dict]:
    sql = (sql_dir / "11_query_colbert.sql").read_text()
    candidate_pool = top_k * 3
    chunk_scores: dict = {}
    conn = get_connection()
    cur  = conn.cursor()
    try:
        for token_vec in q_colbert:
            vec_str = f"[{','.join(map(str, token_vec.tolist()))}]"
            cur.execute(sql, (vec_str, candidate_pool))
            for chunk_id, chunk_text, max_sim in cur.fetchall():
                entry = chunk_scores.setdefault(chunk_id, {"chunk_text": chunk_text, "sims": []})
                entry["sims"].append(float(max_sim))
    finally:
        cur.close(); conn.close()
    results = [
        {"chunk_id": cid, "chunk_text": d["chunk_text"], "score": float(np.mean(d["sims"]))}
        for cid, d in chunk_scores.items()
    ]
    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict]],
    top_k: int,
    k: int = 60,
) -> list[dict]:
    """Merge ranked lists using RRF and return top_k results."""
    rrf_scores: dict = {}
    chunk_texts: dict = {}
    for results in ranked_lists:
        for rank, item in enumerate(results, start=1):
            cid = item["chunk_id"]
            rrf_scores[cid]  = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank)
            chunk_texts[cid] = item["chunk_text"]
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [
        {"chunk_id": cid, "chunk_text": chunk_texts[cid], "rrf_score": score}
        for cid, score in fused
    ]


def ensemble_retrieve(query: str, top_k: int) -> list[dict]:
    """Embed query, run all three retrievers, fuse with RRF."""
    q_out = embed_model.encode(
        [query],
        max_length=config.embedding.max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True,
    )
    ranked_lists = [
        retrieve_dense(q_dense=q_out["dense_vecs"][0], top_k=top_k),
        retrieve_sparse(q_sparse=q_out["lexical_weights"][0], top_k=top_k),
        retrieve_colbert(q_colbert=q_out["colbert_vecs"][0], top_k=top_k),
    ]
    return reciprocal_rank_fusion(ranked_lists=ranked_lists, top_k=top_k)


print("✅ Retrieval functions defined")

---

## 2. Re-ranking Techniques

We implement and compare **three** re-ranking strategies:

| Technique | How it works | Cost |
|---|---|---|
| **Cross-encoder** | Deep `(query, doc)` model, highest accuracy | ~100 ms per candidate |
| **MMR** | Balances relevance *and* diversity to avoid redundant chunks | Zero extra model calls |
| **BGE score re-sort** | Uses the dense cosine score already computed at retrieval | Free — already have it |

### 2.1 Cross-Encoder Re-ranking

A cross-encoder jointly encodes the query and each candidate document. Because both are processed together, it can model fine-grained interactions between query tokens and document tokens — something bi-encoders cannot do.

In [ ]:
def rerank_cross_encoder(
    query: str,
    chunks: list[dict],
    top_n: int,
) -> list[dict]:
    """
    Re-rank chunks using a BGE cross-encoder via transformers directly.

    Args:
        query  : User question.
        chunks : Candidate chunks from the retriever.
        top_n  : How many to keep after re-ranking.

    Returns:
        Top-n chunks with an added 'rerank_score' key.
    """
    pairs = [[query, c["chunk_text"]] for c in chunks]
    inputs = reranker_tokenizer(
        pairs,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    with torch.no_grad():
        scores = reranker_model(**inputs).logits.squeeze(-1)
        scores = torch.sigmoid(scores).tolist()

    if isinstance(scores, float):
        scores = [scores]

    for chunk, score in zip(chunks, scores):
        chunk["rerank_score"] = float(score)

    return sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)[:top_n]

### 2.2 Maximal Marginal Relevance (MMR)

MMR iteratively selects the chunk that is **most relevant to the query** and **least similar to already-selected chunks**. This avoids the common failure mode of retrieving five nearly-identical passages.

$$\text{MMR}(d) = \lambda \cdot \text{sim}(d, q) - (1 - \lambda) \cdot \max_{d' \in S} \text{sim}(d, d')$$

where $S$ is the set of already-selected chunks and $\lambda$ trades off relevance vs. diversity.

In [ ]:
def rerank_mmr(
    query: str,
    chunks: list[dict],
    top_n: int,
    lambda_param: float = 0.6,
) -> list[dict]:
    """
    Re-rank chunks with Maximal Marginal Relevance.

    Args:
        query        : User question.
        chunks       : Candidate chunks from the retriever.
        top_n        : How many to keep after re-ranking.
        lambda_param : 1.0 = pure relevance, 0.0 = pure diversity.

    Returns:
        Top-n chunks re-ordered by MMR score.
    """
    texts = [c["chunk_text"] for c in chunks]
    all_texts = [query] + texts

    # Embed everything in one batch
    vecs = embed_model.encode(
        all_texts,
        max_length=config.embedding.max_length,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )["dense_vecs"]

    q_vec   = vecs[0]          # query embedding
    doc_vecs = vecs[1:]         # document embeddings

    # Cosine similarities: query–doc and doc–doc
    query_sims = doc_vecs @ q_vec                      # (n,)
    doc_sims   = doc_vecs @ doc_vecs.T                 # (n, n)

    selected: list[int] = []
    remaining = list(range(len(chunks)))

    while len(selected) < top_n and remaining:
        if not selected:
            # First pick: highest query similarity
            best = max(remaining, key=lambda i: query_sims[i])
        else:
            best = max(
                remaining,
                key=lambda i: (
                    lambda_param * query_sims[i]
                    - (1 - lambda_param) * max(doc_sims[i][j] for j in selected)
                ),
            )
        selected.append(best)
        remaining.remove(best)

    return [chunks[i] for i in selected]

### 2.3 Score-Based Re-sort

As a simple baseline, we re-sort by the **dense cosine score** that was already computed during retrieval. This is essentially "use only the dense retriever's ranking" and costs nothing extra. It serves as a useful lower bound to compare against.

In [ ]:
def rerank_by_dense_score(
    chunks: list[dict],
    top_n: int,
) -> list[dict]:
    """
    Baseline re-ranker: sort by the RRF score already in the chunk dict.

    Args:
        chunks : Candidate chunks from ensemble retrieval.
        top_n  : How many to keep.

    Returns:
        Top-n chunks sorted by rrf_score.
    """
    return sorted(chunks, key=lambda x: x["rrf_score"], reverse=True)[:top_n]


print("✅ All re-ranking functions defined")

---

## 3. LangChain RAG Chain

The chain is identical to Notebook 5. We define it once and inject different re-ranked contexts at call time.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using only the provided context.\n"
    "If the answer is not in the context, say \"I don't know based on the provided context.\"\n"
    "Be concise and accurate."
)

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=config.groq.model,
    temperature=config.groq.temperature,
    max_tokens=config.groq.max_tokens,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user",   "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {
        "context":  RunnableLambda(lambda x: x["context"]),
        "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"]),
    }
    | prompt
    | llm
    | StrOutputParser()
)


def format_context(chunks: list[dict]) -> str:
    """Join retrieved chunks into a single prompt-ready context string."""
    return "\n\n---\n\n".join(
        f"[Chunk {i + 1}]\n{c['chunk_text']}" for i, c in enumerate(chunks)
    )


print("✅ LangChain RAG chain ready")

---

## 4. Compare Re-ranking Strategies

We run the same question through all three re-rankers (and a no-rerank baseline) and compare which chunks each strategy promotes to the top.

In [ ]:
QUESTION = "Why did Alice assume the March Hare wouldn't be ""raving mad"" during her visit?"

# Retrieve a larger pool so the re-ranker has real choices
CANDIDATE_POOL = config.retrieval.top_k * config.reranker.candidate_factor
FINAL_TOP_N    = config.retrieval.top_k

print(f"Retrieving {CANDIDATE_POOL} candidate chunks …")
candidates = ensemble_retrieve(query=QUESTION, top_k=CANDIDATE_POOL)
print(f"✅ {len(candidates)} candidates retrieved\n")

In [ ]:
import copy

# Deep-copy so each ranker works on a fresh list
results_baseline      = rerank_by_dense_score(chunks=copy.deepcopy(candidates), top_n=FINAL_TOP_N)
results_cross_encoder = rerank_cross_encoder(query=QUESTION, chunks=copy.deepcopy(candidates), top_n=FINAL_TOP_N)
results_mmr           = rerank_mmr(query=QUESTION, chunks=copy.deepcopy(candidates), top_n=FINAL_TOP_N)

print("=" * 70)
print(f"QUESTION: {QUESTION}")
print("=" * 70)

for label, chunks in [
    ("Baseline (RRF order)",  results_baseline),
    ("Cross-Encoder",          results_cross_encoder),
    ("MMR",                    results_mmr),
]:
    print(f"\n── {label} ──")
    for rank, c in enumerate(chunks, 1):
        snippet = c["chunk_text"][:120].replace("\n", " ")
        print(f"  [{rank}] {snippet}…")

---

## 5. End-to-End Answers

Now let's pass each re-ranked context to the LLM and compare the answers side by side.

In [ ]:
def ask_with_chunks(question: str, chunks: list[dict]) -> str:
    """Run the RAG chain with a pre-ranked list of chunks."""
    return rag_chain.invoke(
        {"question": question, "context": format_context(chunks=chunks)}
    )


answer_baseline      = ask_with_chunks(question=QUESTION, chunks=results_baseline)
answer_cross_encoder = ask_with_chunks(question=QUESTION, chunks=results_cross_encoder)
answer_mmr           = ask_with_chunks(question=QUESTION, chunks=results_mmr)

print("=" * 70)
print(f"QUESTION: {QUESTION}")
print("=" * 70)

for label, answer in [
    ("Baseline (RRF order)",  answer_baseline),
    ("Cross-Encoder",          answer_cross_encoder),
    ("MMR",                    answer_mmr),
]:
    print(f"\n── {label} ──")
    print(textwrap.fill(answer, width=70))

---

## 6. Try Your Own Question

Change the question below to explore the difference between re-rankers on other parts of the book.

In [ ]:
# MY_QUESTION = "In the Mouse's long tale (shaped like a tail), who was the character that proposed going to law and acting as both judge and jury?"
# MY_QUESTION = "What was the precise first sentence in Alice’s French lesson-book that she used to address the Mouse?"
MY_QUESTION = "What geographically incorrect word did Alice use when trying to think of the word 'Antipodes' to describe the people who walk with their heads downward?"

my_candidates = ensemble_retrieve(query=MY_QUESTION, top_k=CANDIDATE_POOL)
my_chunks     = rerank_cross_encoder(
    query=MY_QUESTION,
    chunks=my_candidates,
    top_n=FINAL_TOP_N,
)
my_answer = ask_with_chunks(question=MY_QUESTION, chunks=my_chunks)

print(f"❓ {MY_QUESTION}\n")
print(f"🤖 {my_answer}")

---

## 7. Summary

| Re-ranker | Strength | Weakness |
|---|---|---|
| **Baseline (RRF)** | Zero extra cost | Ignores fine-grained query–doc interactions |
| **Cross-Encoder** | Highest relevance precision | Slower (one model call per candidate) |
| **MMR** | Reduces redundancy in context | May deprioritise highly relevant but similar chunks |

### Key takeaways

- Retrieve a **larger pool** (e.g. 3× final top-k) and re-rank down — this is the standard two-stage pattern.
- **Cross-encoder** re-ranking consistently improves precision over pure bi-encoder retrieval.
- **MMR** is most useful when retrieved chunks tend to be repetitive (e.g. the same event described across multiple nearby paragraphs).
- In the next notebook we will **measure** these differences quantitatively using RAGAS.